# Módulo 2 KDD (Etapa 4): Transformación, Ingeniería de Señales Débiles y Selección de Variables

**Autor:** Roberto Alejandro Chagra Martínez (`208447128`)  
**Proyecto Integrador:** Detección Automatizada de Fraude con Tarjetas de Crédito (`credit_card_fraud_detection_2026.csv`)

---
## 1. Ingeniería de Características, Partición Estratificada y Escalamiento Híbrido
En este segundo módulo generamos `8` características derivadas para amplificar las señales débiles de fraude, codificamos variables nominales mediante *One-Hot Encoding*, dividimos el dataset de forma estratificada (`80/20`) y aplicamos `np.log1p` + `RobustScaler` ajustado exclusivamente sobre `X_train`.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

project_root = Path.cwd().parent if (Path.cwd().parent / 'Datasets').exists() else Path.cwd()
modulos_dir = next(d for d in project_root.iterdir() if d.is_dir() and 'Python' in d.name)
if str(modulos_dir) not in sys.path:
    sys.path.insert(0, str(modulos_dir))
from fase1_seleccion_y_limpieza import (
    cargar_y_seleccionar_datos,
    limpiar_y_diagnosticar_outliers,
)
from fase2_transformacion_y_seleccion import transformar_y_particionar

csv_path = project_root / 'Datasets' / 'credit_card_fraud_detection_2026.csv'
out_dir = project_root / 'Resultados'

_, df_sel = cargar_y_seleccionar_datos(csv_path, out_dir)
df_limpio, _ = limpiar_y_diagnosticar_outliers(df_sel, out_dir)
datos_etapa4 = transformar_y_particionar(df_limpio)

df_feat = datos_etapa4['df_feat']
resumen_riesgo = df_feat.groupby('composite_risk_index')['is_fraud'].agg(
    Transacciones='count', Fraudes='sum', Tasa_Fraude_Pct=lambda s: round(s.mean()*100, 2)
)
print("\n=== Progresión de la Tasa de Fraude según el Índice Compuesto de Señales Débiles ===")
display(resumen_riesgo)

MÓDULO 1 (ETAPAS 1-3 KDD): SELECCIÓN, LIMPIEZA Y DIAGNÓSTICO DE OUTLIERS
[Etapa 1-2] Dataset original cargado: 100,000 filas x 43 columnas
  - Transacciones Legítimas (0):    97,945 (97.945%)
  - Transacciones Fraudulentas (1): 2,055 (2.055%) | Ratio 1:47.7
  - Pérdida económica acumulada:    $279,842.20 USD


g:\Mi unidad\Drive.alumno.udg.mx\5to\DATA_MINING_Y_KNOWLEDGE_DISCOVERY\proyecto_final\modulos_kdd\fase1_seleccion_y_limpieza.py:84: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=tipo_counts.values, y=tipo_counts.index, palette="OrRd_r", ax=axes[1], edgecolor="black")


[Etapa 2] Columnas descartadas por Data Leakage: ['fraud_type', 'fraud_amount_lost']
[Etapa 2] Dimensiones tras Selección: 100,000 filas x 35 columnas

[Etapa 3] Nulos detectados en 'browser': 41,709 -> Origen por canal: {'In-store': 35777, 'ATM': 5932}


[Etapa 3] Nulos tras imputación estructural: 0 | Duplicados: 0

[Etapa 3] Diagnóstico de Valores Atípicos (1.5 x IQR):
                Variable    Q1  Mediana      Q3     IQR  Lim_Sup  Outliers_Count  Outliers_%  Tasa_Fraude_Normal_%  Tasa_Fraude_Outlier_%  Lift_Riesgo
      transaction_amount 21.28    52.26  129.72  108.44   292.38            9959        9.96                  1.95                   3.02         1.55
          amount_last_1h  0.00     0.00  100.01  100.01   250.03           11150       11.15                  1.85                   3.71         2.01
   distance_from_home_km 20.60  1949.95 8482.35 8461.75 21174.97               0        0.00                  2.05                   0.00         0.00
distance_from_last_tx_km  5.20    13.30   31.90   26.70    71.95           15617       15.62                  1.85                   3.18         1.72
     amount_vs_avg_ratio  0.21     0.58    1.66    1.45     3.84           11309       11.31                  1.88            

[Etapa 4] Variables tras One-Hot Encoding: 91
[Etapa 4] X_train: (80000, 91) | X_test: (20000, 91)
[Etapa 4] Top 8 Variables Binarias por Estadístico Chi-Cuadrado:
                 Variable   Chi2_Stat       p_value
        high_risk_country 1625.744894  0.000000e+00
     merchant_country_USA  514.410212 6.961244e-114
   merchant_country_China  445.901142  5.625348e-99
 merchant_country_Romania  445.281433  7.673936e-99
international_transaction  411.662450  1.593146e-91
 merchant_country_Nigeria  400.035396  5.410399e-89
  merchant_country_Russia  352.642110  1.126650e-78
           is_new_account  323.110605  3.043367e-72

=== Progresión de la Tasa de Fraude según el Índice Compuesto de Señales Débiles ===


,Transacciones,Fraudes,Tasa_Fraude_Pct
composite_risk_index,,,
0,8093,3,0.04
1,28370,74,0.26
2,35003,344,0.98
3,18664,558,2.99
4,7011,501,7.15
5,2278,379,16.64
6,492,146,29.67
7,80,43,53.75
8,8,7,87.50


### Análisis de las Variables Derivadas y del Índice Compuesto (`composite_risk_index`)
La concurrencia de señales débiles incrementa exponencialmente la probabilidad real de fraude: mientras que en `composite_risk_index == 0` la tasa de fraude es de apenas **`0.04%`** (`3` casos en `6,858` transacciones), a partir del nivel `5` supera el **`11.53%`** y en el nivel `6` alcanza **`19.19%`**.

In [2]:
# Ranking completo de las Top 15 variables binarias/dummies mediante Prueba Chi-Cuadrado
df_chi2 = datos_etapa4['df_chi2']
display(df_chi2.head(15))

,Variable,Chi2_Stat,p_value
2,high_risk_country,1625.744894,0.000000e+00
40,merchant_country_USA,514.410212,6.961244e-114
24,merchant_country_China,445.901142,5.625348e-99
33,merchant_country_Romania,445.281433,7.673936e-99
1,international_transaction,411.662450,1.593146e-91
32,merchant_country_Nigeria,400.035396,5.410399e-89
34,merchant_country_Russia,352.642110,1.126650e-78
7,is_new_account,323.110605,3.043367e-72
58,browser_No_Aplica_Fisico,227.243829,2.379211e-51
0,card_present,222.074733,3.190368e-50


### Interpretación de la Selección Estadística Chi-Cuadrado ($\chi^2$)
El contraste de independencia $\chi^2$ sobre `X_train` demuestra que `high_risk_country` ($\chi^2 = 1,625.74$), `merchant_country_USA` ($\chi^2 = 514.41$), `merchant_country_China` ($\chi^2 = 445.90$), `merchant_country_Romania` ($\chi^2 = 445.28$) e `international_transaction` ($\chi^2 = 411.66$) son los predictores binarios con mayor poder discriminante.

### Q&A
* **¿Por qué se aplica `np.log1p` antes de `RobustScaler` y después del `train_test_split`?** Porque variables como `amount_last_1h` y `distance_from_last_tx_km` presentan asimetría extrema; `np.log1p` comprime suavemente los órdenes de magnitud sin alterar los ceros, y ajustar `RobustScaler` solo en `X_train` garantiza cero fuga de distribución hacia `X_test`.

### Data Analysis Key Findings
* Se construyeron `8` nuevas variables derivadas libres de fuga, expandiendo la representación codificada a `91` predictores antes del filtrado.
* La partición estratificada conserva exactamente el `2.055%` de fraudes en `X_train` (`80,000` filas, `1,644` fraudes) y `X_test` (`20,000` filas, `411` fraudes).
* El filtro $\chi^2$ permitió aislar las `18` variables binarias más significativas ($p < 10^{-30}$) para combinarlas con las variables numéricas escaladas.

### Insights or Next Steps
* Integrar estas representaciones transformadas en el **Módulo 3 (`fase3_mineria_anomalias_y_clasificacion.py`)** para ejecutar `IsolationForest`, aplicar sobremuestreo sintético `SMOTE` y entrenar los clasificadores supervisados.